# Tema Programare: Regresie Logistica - Ghid Complet

Bun venit la tema de programare despre Regresie Logistica!

Regresia logistica este un algoritm fundamental de clasificare folosit atunci cand variabila tinta este categoriala. Spre deosebire de regresia liniara, care prezice valori continue, regresia logistica prezice probabilitati si etichete de clasa. Este folosita pe scara larga in probleme de clasificare binara precum detectarea spam-ului, diagnosticarea bolilor si predictia churn-ului clientilor.

**Ce vei face in aceasta tema:**

* **Clasificare binara** - Implementezi regresie logistica pentru probleme cu doua clase
* **Interpretezi odds ratio** - Intelegi cum afecteaza trasaturile probabilitatea clasei pozitive
* **Analizezi frontiera de decizie** - Vizualizezi cum separa modelul clasele
* **Clasificare multi-clasa** - Extinzi la probleme cu mai mult de doua clase
* **Regresie logistica regularizata** - Aplici regularizare L1 si L2
* **Implementare de la zero** - Construiesti regresia logistica de la zero cu gradient descent

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul. **Nu adauga si nu modifica cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, asa ca nu te baza pe ele pentru solutia finala. Foloseste spatiile deja oferite in notebook.

* Evita variabilele globale daca nu sunt absolut necesare. Evaluatorul iti testeaza codul intr-un mediu izolat, fara sa ruleze neaparat toate celulele de sus in jos. Din acest motiv, variabilele globale pot sa nu fie disponibile in momentul evaluarii. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Inainte sa trimiti notebook-ul pentru evaluare, salveaza-l mai intai folosind iconita 💾 din coltul stanga-sus al paginii, apoi apasa butonul `Submit assignment` din coltul dreapta-sus.
---


## Cuprins
- [Importuri](#imports)
- [1 - Regresie logistica binara](#1)
    - [1.1 - Incarca si exploreaza datele](#1-1)
    - [1.2 - Implementeaza regresia logistica binara](#1-2)
        - **[Exercitiul 1 - binary_logistic_regression](#ex-1)**
    - [1.3 - Evalueaza performanta modelului](#1-3)
        - **[Exercitiul 2 - compute_classification_metrics](#ex-2)**
    - [1.4 - Interpreteaza odds ratios](#1-4)
        - **[Exercitiul 3 - compute_odds_ratios](#ex-3)**
- [2 - Analiza granitei de decizie](#2)
    - [2.1 - Vizualizeaza granita de decizie](#2-1)
    - [2.2 - Granita de decizie cu doua trasaturi](#2-2)
        - **[Exercitiul 4 - plot_decision_boundary](#ex-4)**
- [3 - Clasificare multi-clasa](#3)
    - [3.1 - Strategia One-vs-Rest](#3-1)
        - **[Exercitiul 5 - multiclass_logistic_regression](#ex-5)**
    - [3.2 - Analiza matricei de confuzie](#3-2)
- [4 - Regresie logistica regularizata](#4)
    - [4.1 - Regularizare L2 (Ridge)](#4-1)
        - **[Exercitiul 6 - l2_logistic_regression](#ex-6)**
    - [4.2 - Regularizare L1 (LASSO)](#4-2)
        - **[Exercitiul 7 - l1_logistic_regression](#ex-7)**
    - [4.3 - Compara efectele regularizarii](#4-3)
- [5 - Implementare de la zero](#5)
    - [5.1 - Implementeaza functia sigmoid](#5-1)
        - **[Exercitiul 8 - sigmoid](#ex-8)**
    - [5.2 - Implementeaza functia de cost](#5-2)
        - **[Exercitiul 9 - compute_cost_logistic](#ex-9)**
    - [5.3 - Implementeaza gradient descent](#5-3)
        - **[Exercitiul 10 - gradient_descent_logistic](#ex-10)**
    - [5.4 - Antreneaza si compara](#5-4)


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')

<a name='1'></a>
## 1 - Regresie logistica binara

Regresia logistica modeleaza probabilitatea ca o instanta sa apartina unei anumite clase. Pentru clasificare binara:

$$P(y=1|x) = \sigma(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

unde $\sigma$ este functia sigmoid. Modelul returneaza probabilitati pe care apoi le convertim in etichete de clasa.


<a name='1-1'></a>
### 1.1 - Incarca si exploreaza datele

Vom folosi un set de date in care prezicem daca pretul unei case este peste sau sub un prag, pe baza dimensiunii sale.


In [ ]:
# Load the dataset
data = pd.read_csv('ex1data1.csv')
print("Dataset shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())

# Convert to binary classification problem
THRESHOLD = data.iloc[:, 1].median()
print(f"\nUsing threshold: {THRESHOLD:.2f}")

X_binary = data.iloc[:, 0].values.reshape(-1, 1)
y_binary = (data.iloc[:, 1].values >= THRESHOLD).astype(int)

print(f"\nClass distribution:")
print(f"  Class 0: {np.sum(y_binary == 0)} samples")
print(f"  Class 1: {np.sum(y_binary == 1)} samples")

In [ ]:
# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X_binary[y_binary == 0], y_binary[y_binary == 0], 
            alpha=0.6, color='red', edgecolors='k', s=100, label='Class 0')
plt.scatter(X_binary[y_binary == 1], y_binary[y_binary == 1], 
            alpha=0.6, color='blue', edgecolors='k', s=100, label='Class 1')
plt.xlabel('Feature (Population)', fontsize=12)
plt.ylabel('Class Label', fontsize=12)
plt.title('Binary Classification Data', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.yticks([0, 1])
plt.show()

<a name='1-2'></a>
### 1.2 - Implementeaza regresia logistica binara
<a name='ex-1'></a>
**Exercitiul 1 - binary_logistic_regression**

Implementeaza o functie care antreneaza un model de regresie logistica binara folosind scikit-learn.

**Instructiuni:**
- Foloseste `LogisticRegression`.
- Antreneaza modelul pe datele de intrare.
- Returneaza modelul antrenat si predictiile sale.


In [ ]:
def binary_logistic_regression(X, y):
    """
    Fit a binary logistic regression model.
    
    Args:
        X: Feature array of shape (m, n)
        y: Binary target array of shape (m,)
    
    Returns:
        model: Fitted LogisticRegression model
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 2 lines)
    model = LogisticRegression()
    model.fit(X, y)
    ### SFÂRȘIT DE COD AICI ###
    
    return model

In [ ]:
# Test your implementation
model_binary = binary_logistic_regression(X_binary, y_binary)
print(f"Model coefficients (weights): {model_binary.coef_}")
print(f"Model intercept (bias): {model_binary.intercept_}")

# Make predictions
y_pred = model_binary.predict(X_binary)
y_pred_proba = model_binary.predict_proba(X_binary)[:, 1]

print(f"\nSample predictions (first 10):")
print(f"  True labels: {y_binary[:10]}")
print(f"  Predicted labels: {y_pred[:10]}")
print(f"  Predicted probabilities: {y_pred_proba[:10].round(3)}")


In [ ]:
unittests.exercise_1(binary_logistic_regression)

<a name='1-3'></a>
### 1.3 - Evalueaza performanta modelului
<a name='ex-2'></a>
**Exercitiul 2 - compute_classification_metrics**

Implementeaza o functie care calculeaza metrici esentiale de clasificare:
- **Accuracy**: fractia predictiilor corecte;
- **Precision**: dintre predictiile pozitive, cate sunt corecte;
- **Recall**: dintre exemplele pozitive reale, cate au fost detectate;
- **F1-score**: media armonica intre precision si recall.


In [ ]:
def compute_classification_metrics(y_true, y_pred):
    """
    Compute classification performance metrics.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        metrics: Dictionary with accuracy, precision, recall, f1
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred)
    }
    ### SFÂRȘIT DE COD AICI ###
    
    return metrics

In [ ]:
# Test your implementation
metrics = compute_classification_metrics(y_binary, y_pred)

print("Classification Metrics:")
for metric_name, value in metrics.items():
    print(f"  {metric_name.capitalize()}: {value:.4f}")

# Display confusion matrix
cm = confusion_matrix(y_binary, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix', fontsize=14)
plt.show()


In [ ]:
unittests.exercise_2(compute_classification_metrics)

<a name='1-4'></a>
### 1.4 - Interpreteaza odds ratios
<a name='ex-3'></a>
**Exercitiul 3 - compute_odds_ratios**

Odds ratio ne ajuta sa interpretam modul in care fiecare trasatura influenteaza probabilitatea clasei pozitive.

Pentru un coeficient $\theta_j$, odds ratio-ul este:
$$OR_j = e^{\theta_j}$$


In [ ]:
def compute_odds_ratios(model, feature_names=None):
    """
    Compute odds ratios for logistic regression coefficients.
    
    Args:
        model: Fitted LogisticRegression model
        feature_names: List of feature names (optional)
    
    Returns:
        odds_ratios: Dictionary mapping features to odds ratios
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    coefficients = model.coef_.flatten()
    
    if feature_names is None:
        feature_names = [f"Feature_{i}" for i in range(len(coefficients))]
    
    odds_ratios = dict(zip(feature_names, np.exp(coefficients)))
    ### SFÂRȘIT DE COD AICI ###
    
    return odds_ratios

In [ ]:
# Test your implementation
odds_ratios = compute_odds_ratios(model_binary, ['Population'])

print("Odds Ratios:")
for feature, or_value in odds_ratios.items():
    if or_value > 1:
        interpretation = f"increases odds by {((or_value - 1) * 100):.2f}%"
    else:
        interpretation = f"decreases odds by {((1 - or_value) * 100):.2f}%"
    print(f"  {feature}: {or_value:.4f} ({interpretation})")


In [ ]:
unittests.exercise_3(compute_odds_ratios)

<a name='2'></a>
## 2 - Analiza granitei de decizie

Granita de decizie este locul in care modelul prezice probabilitati egale pentru cele doua clase, adica $P(y=1)=0.5$.


<a name='2-1'></a>
### 2.1 - Vizualizeaza granita de decizie

Sa vizualizam probabilitatile prezise si granita de decizie.


In [ ]:
# Create a range of values for plotting
X_range = np.linspace(X_binary.min(), X_binary.max(), 300).reshape(-1, 1)
y_proba_range = model_binary.predict_proba(X_range)[:, 1]

# Plot probability curve
plt.figure(figsize=(12, 6))

# Subplot 1: Probability curve
plt.subplot(1, 2, 1)
plt.plot(X_range, y_proba_range, linewidth=2, color='green', label='P(y=1|x)')
plt.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Decision Boundary (0.5)')
plt.scatter(X_binary, y_binary, alpha=0.4, c=y_binary, cmap='coolwarm', edgecolors='k')
plt.xlabel('Feature Value', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title('Logistic Regression: Probability Curve', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Decision regions
plt.subplot(1, 2, 2)
y_pred_range = (y_proba_range >= 0.5).astype(int)
plt.scatter(X_range, y_pred_range, c=y_pred_range, cmap='coolwarm', alpha=0.3, s=10)
plt.scatter(X_binary, y_binary, c=y_binary, cmap='coolwarm', alpha=0.8, edgecolors='k', s=100)
plt.xlabel('Feature Value', fontsize=12)
plt.ylabel('Predicted Class', fontsize=12)
plt.title('Decision Regions', fontsize=14)
plt.yticks([0, 1])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<a name='2-2'></a>
### 2.2 - Granita de decizie cu doua trasaturi
<a name='ex-4'></a>
**Exercitiul 4 - plot_decision_boundary**

Cand avem doua trasaturi, putem vizualiza granita de decizie ca o linie sau curba in spatiu 2D. Implementeaza functia care afiseaza aceasta frontiera pe baza unui model antrenat.


In [ ]:
# First, create a 2D dataset
data_2d = pd.read_csv('ex1data2.csv')
X_2d = data_2d.iloc[:, :2].values
y_2d = (data_2d.iloc[:, 2].values >= data_2d.iloc[:, 2].median()).astype(int)

# Standardize features
scaler = StandardScaler()
X_2d_scaled = scaler.fit_transform(X_2d)

# Fit logistic regression
model_2d = binary_logistic_regression(X_2d_scaled, y_2d)

print("2D Model trained successfully!")
print(f"Training accuracy: {model_2d.score(X_2d_scaled, y_2d):.4f}")

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """
    Plot decision boundary for a 2D classification problem.
    
    Args:
        model: Fitted classifier
        X: Feature array of shape (m, 2)
        y: Binary target array
        title: Plot title
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 15 lines)
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.25)

    y = np.asarray(y).ravel()
    plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Class 0", edgecolors="k", s=40)
    plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Class 1", edgecolors="k", s=40)
    ### SFÂRȘIT DE COD AICI ###

    plt.xlabel('Feature 1 (scaled)', fontsize=12)
    plt.ylabel('Feature 2 (scaled)', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend()
    plt.show()

In [ ]:
# Test your implementation
plot_decision_boundary(model_2d, X_2d_scaled, y_2d, "Logistic Regression: 2D Decision Boundary")


In [ ]:
unittests.exercise_4(plot_decision_boundary)

<a name='3'></a>
## 3 - Clasificare multi-clasa

Regresia logistica poate fi extinsa la probleme multi-clasa folosind strategii precum One-vs-Rest (OvR).


<a name='3-1'></a>
### 3.1 - Strategia One-vs-Rest
<a name='ex-5'></a>
**Exercitiul 5 - multiclass_logistic_regression**

Implementeaza regresie logistica multi-clasa folosind scikit-learn.

**Instructiuni:**
- Foloseste `LogisticRegression` cu o configuratie potrivita pentru multi-clasa.
- Antreneaza modelul.
- Returneaza modelul si predictiile sale.


In [ ]:
# Create multi-class dataset
def price_to_class(price, thresholds=[120, 160, 200]):
    for i, threshold in enumerate(thresholds):
        if price < threshold:
            return i
    return len(thresholds)

data_multi = pd.read_csv('ex1data1.csv')
X_multi = data_multi.iloc[:, 0].values.reshape(-1, 1)
prices = data_multi.iloc[:, 1].values
y_multi = np.array([price_to_class(p, [2, 4, 6]) for p in prices])

print(f"Multi-class dataset created:")
print(f"  Number of classes: {len(np.unique(y_multi))}")
print(f"  Class distribution: {np.bincount(y_multi)}")

In [ ]:
def multiclass_logistic_regression(X, y):
    """
    Fit a multi-class logistic regression model using One-vs-Rest.
    
    Args:
        X: Feature array
        y: Multi-class target array
    
    Returns:
        model: Fitted LogisticRegression model
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 2 lines)
    model = None  # Hint: set multi_class='ovr'
    ### SFÂRȘIT DE COD AICI ###
    
    return model

In [ ]:
# Test your implementation
model_multi = multiclass_logistic_regression(X_multi, y_multi)
y_pred_multi = model_multi.predict(X_multi)

accuracy = accuracy_score(y_multi, y_pred_multi)
print(f"Multi-class classification accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_multi, y_pred_multi, zero_division=0))

In [ ]:
unittests.exercise_5(multiclass_logistic_regression)

<a name='3-2'></a>
### 3.2 - Analiza matricei de confuzie

Pentru problemele multi-clasa, matricile de confuzie sunt foarte utile pentru a vedea ce clase sunt confundate intre ele.


In [ ]:
# Plot confusion matrix for multi-class
cm_multi = confusion_matrix(y_multi, y_pred_multi)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Class {i}' for i in range(len(np.unique(y_multi)))],
            yticklabels=[f'Class {i}' for i in range(len(np.unique(y_multi)))])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Multi-class Confusion Matrix', fontsize=14)
plt.show()

<a name='4'></a>
## 4 - Regresie logistica regularizata

Regularizarea previne supraantrenarea prin penalizarea coeficientilor mari.

**L2 (Ridge):**
$$J(\theta) = -\frac{1}{m}\sum[y\log(h) + (1-y)\log(1-h)] + \frac{\lambda}{2m}\sum_{j=1}^{n}\theta_j^2$$

**L1 (LASSO):**
$$J(\theta) = -\frac{1}{m}\sum[y\log(h) + (1-y)\log(1-h)] + \frac{\lambda}{m}\sum_{j=1}^{n}|\theta_j|$$


<a name='4-1'></a>
### 4.1 - Regularizare L2 (Ridge)
<a name='ex-6'></a>
**Exercitiul 6 - l2_logistic_regression**

Implementeaza regresie logistica regularizata L2.

**Instructiuni:**
- Foloseste `LogisticRegression` cu `penalty='l2'`.
- Antreneaza modelul.
- Returneaza modelul si coeficientii.


In [ ]:
def l2_logistic_regression(X, y, C=1.0):
    """
    Fit L2 regularized logistic regression.
    
    Args:
        X: Feature array
        y: Target array
        C: Inverse of regularization strength
    
    Returns:
        model: Fitted LogisticRegression model
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 2 lines)
    model = None
    ### SFÂRȘIT DE COD AICI ###
    
    return model

In [ ]:
# Test with different C values
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

print("L2 Regularization Results:")
for C in C_values:
    model_l2 = l2_logistic_regression(X_2d_scaled, y_2d, C)
    accuracy = model_l2.score(X_2d_scaled, y_2d)
    coef_norm = np.linalg.norm(model_l2.coef_)
    print(f"  C={C:7.3f}: Accuracy={accuracy:.4f}, ||θ||={coef_norm:.4f}")


In [ ]:
unittests.exercise_6(l2_logistic_regression)

<a name='4-2'></a>
### 4.2 - Regularizare L1 (LASSO)
<a name='ex-7'></a>
**Exercitiul 7 - l1_logistic_regression**

Implementeaza regresie logistica regularizata L1, care poate forta unii coeficienti sa devina exact zero.


In [ ]:
def l1_logistic_regression(X, y, C=1.0):
    """
    Fit L1 regularized logistic regression.
    
    Args:
        X: Feature array
        y: Target array
        C: Inverse of regularization strength
    
    Returns:
        model: Fitted LogisticRegression model
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 2 lines)
    model = None
    ### SFÂRȘIT DE COD AICI ###
    
    return model

In [ ]:
# Test L1 regularization
print("L1 Regularization Results:")
for C in C_values:
    model_l1 = l1_logistic_regression(X_2d_scaled, y_2d, C)
    accuracy = model_l1.score(X_2d_scaled, y_2d)
    n_nonzero = np.sum(model_l1.coef_ != 0)
    coef_norm = np.linalg.norm(model_l1.coef_)
    print(f"  C={C:7.3f}: Accuracy={accuracy:.4f}, Non-zero={n_nonzero}, ||θ||={coef_norm:.4f}")


In [ ]:
unittests.exercise_7(l1_logistic_regression)

<a name='4-3'></a>
### 4.3 - Compara efectele regularizarii

Vizualizeaza cum afecteaza diferit regularizarea L1 si L2 coeficientii.


In [ ]:
# Compare regularization paths
C_range = np.logspace(-3, 3, 50)
l1_coefs = []
l2_coefs = []

for C in C_range:
    model_l1 = l1_logistic_regression(X_2d_scaled, y_2d, C)
    model_l2 = l2_logistic_regression(X_2d_scaled, y_2d, C)
    l1_coefs.append(model_l1.coef_[0])
    l2_coefs.append(model_l2.coef_[0])

l1_coefs = np.array(l1_coefs)
l2_coefs = np.array(l2_coefs)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# L2 coefficients
for i in range(X_2d_scaled.shape[1]):
    ax1.plot(C_range, l2_coefs[:, i], label=f'Feature {i+1}')
ax1.set_xscale('log')
ax1.set_xlabel('C (Inverse Regularization)', fontsize=12)
ax1.set_ylabel('Coefficient Value', fontsize=12)
ax1.set_title('L2 Regularization - Coefficient Paths', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvline(x=1.0, color='red', linestyle='--', alpha=0.5)

# L1 coefficients
for i in range(X_2d_scaled.shape[1]):
    ax2.plot(C_range, l1_coefs[:, i], label=f'Feature {i+1}')
ax2.set_xscale('log')
ax2.set_xlabel('C (Inverse Regularization)', fontsize=12)
ax2.set_ylabel('Coefficient Value', fontsize=12)
ax2.set_title('L1 Regularization - Coefficient Paths', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axvline(x=1.0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\nObservation: L1 can set coefficients to exactly zero (feature selection).")

<a name='5'></a>
## 5 - Implementare de la zero

Acum vom implementa regresia logistica de la zero pentru a intelege cu adevarat algoritmul.


<a name='5-1'></a>
### 5.1 - Implementeaza functia sigmoid
<a name='ex-8'></a>
**Exercitiul 8 - sigmoid**

Functia sigmoid mapeaza orice numar real in intervalul [0, 1]:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$


In [ ]:
def sigmoid(z):
    """
    Compute the sigmoid function.
    
    Args:
        z: Input value (can be scalar, array)
    
    Returns:
        Sigmoid of z
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 2 lines)
    z = np.clip(z, -500, 500)  # For numerical stability
    result = None
    ### SFÂRȘIT DE COD AICI ###
    
    return result

In [ ]:
# Test sigmoid function
z_test = np.array([-10, -5, 0, 5, 10])
print("Sigmoid test:")
print(f"  Input: {z_test}")
print(f"  Output: {sigmoid(z_test)}")

# Plot sigmoid
z_range = np.linspace(-10, 10, 200)
plt.figure(figsize=(10, 6))
plt.plot(z_range, sigmoid(z_range), linewidth=2, color='blue')
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='red', linestyle='--', alpha=0.5)
plt.xlabel('z', fontsize=12)
plt.ylabel('σ(z)', fontsize=12)
plt.title('Sigmoid Function', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
unittests.exercise_8(sigmoid)

<a name='5-2'></a>
### 5.2 - Implementeaza functia de cost
<a name='ex-9'></a>
**Exercitiul 9 - compute_cost_logistic**

Functia de cost pentru entropie incrucisata binara (log loss) este:

$$J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}[y^{(i)}\log(h^{(i)}) + (1-y^{(i)})\log(1-h^{(i)})]$$


In [ ]:
def compute_cost_logistic(X, y, theta, epsilon=1e-15):
    """
    Compute the cost function for logistic regression.
    
    Args:
        X: Feature array of shape (m, n)
        y: Binary target array of shape (m,)
        theta: Parameters of shape (n,)
        epsilon: Small value to prevent log(0)
    
    Returns:
        cost: The cost value (scalar)
    """
    m = len(y)
    
    ### ÎNCEPUT DE COD AICI ### (≈ 4 lines)
    h = None  # Predictions using sigmoid
    h = np.clip(h, epsilon, 1 - epsilon)  # Numerical stability
    
    cost = None  # Binary cross-entropy
    ### SFÂRȘIT DE COD AICI ###
    
    return cost

In [ ]:
# Test cost function
X_test = np.c_[np.ones((len(X_binary), 1)), X_binary]
theta_test = np.zeros(X_test.shape[1])

cost_initial = compute_cost_logistic(X_test, y_binary, theta_test)
print(f"Initial cost (theta=0): {cost_initial:.4f}")
print(f"Expected: ~0.693 (log(2) for balanced classes)")


In [ ]:
unittests.exercise_9(compute_cost_logistic)

<a name='5-3'></a>
### 5.3 - Implementeaza gradient descent
<a name='ex-10'></a>
**Exercitiul 10 - gradient_descent_logistic**

Gradient descent pentru regresia logistica are o regula de actualizare similara:

$$\theta := \theta - \alpha \frac{1}{m} X^T(h - y)$$

Implementeaza aceasta rutina si returneaza parametrii invatati impreuna cu istoricul costului.


In [ ]:
def gradient_descent_logistic(X, y, theta, alpha, num_iters, verbose=True):
    """
    Perform gradient descent for logistic regression.
    
    Args:
        X: Feature array of shape (m, n)
        y: Binary target array of shape (m,)
        theta: Initial parameters of shape (n,)
        alpha: Learning rate
        num_iters: Number of iterations
        verbose: Whether to print progress
    
    Returns:
        theta: Optimized parameters
        cost_history: List of cost values
    """
    m = len(y)
    cost_history = []
    
    for i in range(num_iters):
        ### ÎNCEPUT DE COD AICI ### (≈ 4 lines)
        h = None  # Predictions
        gradient = None  # Gradient
        theta = None  # Update
        
        cost = None  # Compute cost
        ### SFÂRȘIT DE COD AICI ###
        
        cost_history.append(cost)
        
        if verbose and i % 100 == 0:
            print(f"Iteration {i:4d}: Cost = {cost:.4f}")
    
    return theta, cost_history

In [ ]:
# Test gradient descent
theta_initial = np.zeros(X_test.shape[1])
alpha = 0.01
num_iters = 2000

theta_optimized, cost_history = gradient_descent_logistic(
    X_test, y_binary, theta_initial, alpha, num_iters, verbose=True
)

print(f"\nOptimized parameters:")
print(f"  Intercept (θ₀): {theta_optimized[0]:.4f}")
print(f"  Coefficient (θ₁): {theta_optimized[1]:.4f}")


In [ ]:
# Plot cost history
plt.figure(figsize=(10, 6))
plt.plot(cost_history, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost', fontsize=12)
plt.title('Gradient Descent: Cost vs Iteration', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final cost: {cost_history[-1]:.4f}")

In [ ]:
unittests.exercise_10(gradient_descent_logistic)

<a name='5-4'></a>
### 5.4 - Antreneaza si compara

Sa comparam implementarea noastra de la zero cu cea din scikit-learn.


In [ ]:
# Make predictions with from-scratch model
def predict_scratch(X, theta, threshold=0.5):
    probabilities = sigmoid(X @ theta)
    return (probabilities >= threshold).astype(int)

y_pred_scratch = predict_scratch(X_test, theta_optimized)
y_pred_sklearn = model_binary.predict(X_binary)

# Compare accuracies
acc_scratch = accuracy_score(y_binary, y_pred_scratch)
acc_sklearn = accuracy_score(y_binary, y_pred_sklearn)

print("Comparison of From-Scratch vs Scikit-Learn:")
print(f"  From-Scratch:")
print(f"    Intercept: {theta_optimized[0]:.4f}")
print(f"    Coefficient: {theta_optimized[1]:.4f}")
print(f"    Accuracy: {acc_scratch:.4f}")
print(f"\n  Scikit-Learn:")
print(f"    Intercept: {model_binary.intercept_[0]:.4f}")
print(f"    Coefficient: {model_binary.coef_[0][0]:.4f}")
print(f"    Accuracy: {acc_sklearn:.4f}")
print(f"\nDifference in accuracy: {abs(acc_scratch - acc_sklearn):.6f}")

In [ ]:
# Visualize both decision boundaries
X_range = np.linspace(X_binary.min(), X_binary.max(), 300).reshape(-1, 1)
X_range_test = np.c_[np.ones((len(X_range), 1)), X_range]

y_proba_scratch = sigmoid(X_range_test @ theta_optimized)
y_proba_sklearn = model_binary.predict_proba(X_range)[:, 1]

plt.figure(figsize=(12, 6))
plt.plot(X_range, y_proba_scratch, linewidth=2, label='From-Scratch GD', linestyle='--', color='red')
plt.plot(X_range, y_proba_sklearn, linewidth=2, label='Scikit-Learn', alpha=0.7, color='green')
plt.axhline(y=0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
plt.scatter(X_binary, y_binary, alpha=0.4, c=y_binary, cmap='coolwarm', edgecolors='k')
plt.xlabel('Feature Value', fontsize=12)
plt.ylabel('Probability P(y=1|x)', fontsize=12)
plt.title('Comparison: From-Scratch vs Scikit-Learn', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Rezumat si pasii urmatori

**Felicitari!** Ai finalizat tema despre Regresie Logistica. Ai:

- Implementat regresie logistica binara  
- Interpretat odds ratio  
- Analizat frontiere de decizie  
- Extins la clasificare multi-clasa  
- Aplicat regularizare L1 si L2  
- Construit regresia logistica de la zero folosind gradient descent  

**Idei-cheie:**
- Regresia logistica modeleaza probabilitati pentru clasificare
- Sigmoid-ul transforma scorurile liniare in valori intre 0 si 1
- Odds ratio ajuta la interpretarea efectului trasaturilor
- Regularizarea controleaza complexitatea modelului
- Gradient descent optimizeaza functia de cost logistica
